# Train YOLO11n on the ROD-Dataset

**Real-Time Obstacle Detection** — 25 classes, ~24,326 images.
Dataset: https://www.kaggle.com/datasets/abtinzandi/obstacle-detection-dataset

**Code by Ariyan Azami**

> Ultralytics' current default family. File stem is `yolo11n`.

## Before you run (one-time Kaggle setup)
1. **Settings (right sidebar) → Accelerator → `GPU T4 x2`.**
2. **Settings → Internet → ON** (so `ultralytics` installs and pretrained weights download).
3. **+ Add Input → Datasets →** search `obstacle-detection-dataset` by `abtinzandi` and attach it.
   It mounts read-only under `/kaggle/input/...`; the cells below locate it automatically.

Everything is driven by the **Config** cell — edit it and *Run All*.


## 1 · Config — edit me, then Run All

In [ ]:
# ============================ CONFIG ============================
MODEL_VARIANT = "yolov11n"           # which model this notebook trains
WEIGHTS       = "yolo11n.pt"     # pretrained checkpoint to fine-tune from

EPOCHS   = 100        # training length (lower for a quick smoke test)
BATCH    = 128        # TOTAL images per step, split across GPUs (64/GPU on 2x T4)
IMGSZ    = 640        # train/val image size
DEVICE   = "0,1"      # use BOTH T4 GPUs for training (DDP). Set to 0 for one GPU.
EVAL_DEVICE = 0       # val/predict/benchmark always run on a single GPU
PATIENCE = 20         # early-stop if val mAP stalls this many epochs
COS_LR   = True       # cosine LR schedule
SEED     = 42

CONF     = 0.25       # confidence threshold for eval/predict/timing
IOU      = 0.7        # NMS IoU threshold

PROJECT  = "/kaggle/working/runs"
RUN_NAME = f"rod_{MODEL_VARIANT}"
# ===============================================================
print(f"Training {MODEL_VARIANT} from {WEIGHTS} | {EPOCHS} epochs | batch {BATCH} | imgsz {IMGSZ} | device {DEVICE}")

## 2 · Install & environment check

Kaggle's base image usually ships `ultralytics`; with **Internet → ON** we
upgrade it (newer variants like YOLO12/YOLO26 need a recent release). If the
upgrade can't reach PyPI we fall back to the pre-installed version instead of
crashing, and only stop with a clear message if `ultralytics` is truly missing
(in which case enable **Settings → Internet → ON** and re-run).

In [ ]:
import importlib.util, subprocess, sys

# Upgrade when Internet is ON; tolerate offline runs that already have ultralytics.
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"],
                   check=True, timeout=600)
except Exception as e:
    print("ultralytics upgrade skipped (offline or pip error):", e)

if importlib.util.find_spec("ultralytics") is None:
    raise RuntimeError(
        "ultralytics is not installed and could not be fetched. "
        "Enable Settings -> Internet: ON (also required to download pretrained "
        "weights), then Run All again."
    )

In [ ]:
import torch, ultralytics
print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({total/1e9:.0f} GB)")

# Auto-correct DEVICE if fewer GPUs are available than requested (e.g. P100 x1 instead of T4 x2)
if isinstance(DEVICE, str) and "," in DEVICE:
    requested = len(DEVICE.split(","))
    available = torch.cuda.device_count()
    if available < requested:
        DEVICE = ",".join(str(i) for i in range(available)) if available > 0 else "cpu"
        print(f"WARNING: Only {available} GPU(s) available — DEVICE overridden to '{DEVICE}'")

## 3 · Locate the dataset & write a fresh `data.yaml`

`/kaggle/input/` is read-only and the mount path differs per dataset version, so
we search for the dataset's own `data.yaml`, then write a corrected copy into
`/kaggle/working/` with absolute split paths.

In [ ]:
import yaml
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
src = next((p for p in INPUT_ROOT.rglob("data.yaml")), None)
assert src is not None, "data.yaml not found under /kaggle/input — did you Add the dataset?"
data_root = src.parent
print("Dataset root:", data_root)

with open(src) as f:
    orig = yaml.safe_load(f)

data_yaml = {
    "path":  str(data_root),
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    orig["nc"],
    "names": orig["names"],
}
DATA = "/kaggle/working/data.yaml"
with open(DATA, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("Wrote", DATA, "|", data_yaml["nc"], "classes")
print(data_yaml["names"])

## 4 · Train

By default `DEVICE = "0,1"` trains on **both** T4s via PyTorch DDP (`BATCH` is the
total, split across GPUs). If DDP misbehaves in the notebook, set `DEVICE = 0` to
fall back to a single GPU. The dataset is a *detect-segment mixed* set —
Ultralytics keeps the boxes and drops segments automatically (you'll see a
one-time warning, which is expected).

In [ ]:
from ultralytics import YOLO

model = YOLO(WEIGHTS)
results = model.train(
    data     = DATA,
    task     = "detect",
    epochs   = EPOCHS,
    imgsz    = IMGSZ,
    batch    = BATCH,
    device   = DEVICE,
    amp      = True,
    cache    = False,
    workers  = 4,
    project  = PROJECT,
    name     = RUN_NAME,
    patience = PATIENCE,
    cos_lr   = COS_LR,
    plots    = True,
    seed     = SEED,
)
# Multi-GPU DDP training in a notebook relaunches as a subprocess and returns
# None in the parent process, so `results` may be None even though training
# finished. Fall back to locating the run directory under PROJECT/RUN_NAME.
if results is not None and getattr(results, "save_dir", None):
    RUN_DIR = Path(results.save_dir)
else:
    candidates = sorted(Path(PROJECT).glob(RUN_NAME + "*"),
                        key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise RuntimeError(f"No run directory found under {PROJECT}/{RUN_NAME}*")
    RUN_DIR = candidates[-1]
print("Run dir:", RUN_DIR)

## 5 · Validate on the held-out **test** split

We reload `best.pt` (more reliable than reusing the live training object) and
run `val()` on the test split — this reports mAP50, mAP50-95, precision, recall
and writes confusion matrices + PR/F1 curves.

In [ ]:
best = RUN_DIR / "weights" / "best.pt"
eval_model = YOLO(str(best))

metrics = eval_model.val(
    data   = DATA,
    task   = "detect",
    split  = "test",
    imgsz  = IMGSZ,
    batch  = BATCH,
    device = EVAL_DEVICE,   # validate on a single GPU (DDP val in notebooks is flaky)
    conf   = CONF,
    iou    = IOU,
    plots  = True,
)
VAL_DIR = Path(metrics.save_dir)
print(metrics.results_dict)

## 6 · Inference speed benchmark (latency / FPS)

Warm up, then time `predict()` over the whole test set — the same methodology as
the project's `time_inference.py`.

In [ ]:
import time, glob

def benchmark(m, image_paths, imgsz, device=0, conf=0.25, iou=0.7, batch=8, warmup=5):
    for p in image_paths[:warmup]:
        _ = m.predict(p, imgsz=imgsz, device=device, conf=conf, iou=iou, verbose=False)
    t0 = time.perf_counter(); n = 0
    for i in range(0, len(image_paths), batch):
        chunk = image_paths[i:i + batch]
        _ = m.predict(chunk, imgsz=imgsz, device=device, conf=conf, iou=iou, verbose=False)
        n += len(chunk)
    dt = time.perf_counter() - t0
    return (dt / n) * 1000.0, n / dt, dt

test_imgs = sorted(glob.glob(str(data_root / "test/images/*.jpg")))
avg_ms, fps, total_s = benchmark(eval_model, test_imgs, IMGSZ, device=EVAL_DEVICE,
                                 conf=CONF, iou=IOU, batch=BATCH)
print(f"{avg_ms:.2f} ms/img | {fps:.1f} FPS | {total_s:.1f}s over {len(test_imgs)} images")

## 7 · Save a machine-readable summary

Writes `<variant>_summary.json` and `<variant>_results.txt` to
`/kaggle/working/` so the **compare** notebook can chart every model side by
side, and viewers can download the numbers.

In [ ]:
import json
from datetime import datetime

rd = metrics.results_dict
summary = {
    "model":          MODEL_VARIANT,
    "weights":        str(best),
    "timestamp":      datetime.now().isoformat(timespec="seconds"),
    "epochs":         EPOCHS,
    "imgsz":          IMGSZ,
    "batch":          BATCH,
    "params":         int(sum(p.numel() for p in eval_model.model.parameters())),
    "mAP50_95":       rd.get("metrics/mAP50-95(B)"),
    "mAP50":          rd.get("metrics/mAP50(B)"),
    "precision":      rd.get("metrics/precision(B)"),
    "recall":         rd.get("metrics/recall(B)"),
    "avg_ms_per_img": round(avg_ms, 3),
    "fps":            round(fps, 2),
    "test_images":    len(test_imgs),
}
out_json = Path("/kaggle/working") / f"{MODEL_VARIANT}_summary.json"
out_json.write_text(json.dumps(summary, indent=2))
out_txt = Path("/kaggle/working") / f"{MODEL_VARIANT}_results.txt"
out_txt.write_text("\n".join(f"{k}: {v}" for k, v in summary.items()))
print(json.dumps(summary, indent=2))

## 8 · Training & evaluation graphs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show(path, title=""):
    path = Path(path)
    if not path.exists():
        print("[skip]", path.name, "not found"); return
    plt.figure(figsize=(16, 8))
    plt.imshow(mpimg.imread(str(path)))
    plt.axis("off")
    if title:
        plt.title(title, fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

show(RUN_DIR / "results.png",                     "Training curves")
show(VAL_DIR / "confusion_matrix_normalized.png", "Confusion matrix (normalised)")
show(VAL_DIR / "confusion_matrix.png",            "Confusion matrix (raw)")
show(VAL_DIR / "PR_curve.png",                    "Precision-Recall curve")
show(VAL_DIR / "F1_curve.png",                    "F1-Confidence curve")
show(VAL_DIR / "P_curve.png",                     "Precision-Confidence curve")
show(VAL_DIR / "R_curve.png",                     "Recall-Confidence curve")

## 9 · Predictions on sample test images

In [ ]:
import random
random.seed(SEED)
samples = random.sample(test_imgs, min(6, len(test_imgs)))

eval_model.predict(samples, imgsz=IMGSZ, conf=CONF, device=EVAL_DEVICE, save=True,
                   project=PROJECT, name="predict", exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
pred_imgs = sorted(glob.glob(f"{PROJECT}/predict/*.jpg"))[:6]
for ax, p in zip(axes.flat, pred_imgs):
    ax.imshow(mpimg.imread(p)); ax.axis("off")
for ax in axes.flat[len(pred_imgs):]:
    ax.axis("off")
plt.suptitle(f"{MODEL_VARIANT} predictions (conf >= {CONF})", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 10 · Save weights for download

In [ ]:
import shutil
for w in (RUN_DIR / "weights" / "best.pt", RUN_DIR / "weights" / "last.pt"):
    if w.exists():
        dst = Path("/kaggle/working") / f"{MODEL_VARIANT}_{w.name}"
        shutil.copy(str(w), str(dst))
        print("Saved", dst)

## 11 · (Optional) Export for mobile / edge

Uncomment what you need. Mirrors the deployment formats used by the ROD project
(ONNX for desktop, TFLite/NCNN for Android).

In [ ]:
# eval_model.export(format="onnx",  imgsz=IMGSZ, opset=12)
# eval_model.export(format="tflite", imgsz=320, int8=True)
# eval_model.export(format="ncnn",  imgsz=320)